# Adaptive Explainable Hybrid Transformer Framework
## Fine-Grained Waste Classification and Segmentation using Cross-Attention Fusion

This notebook provides a complete interactive walkthrough of the framework, demonstrating how to:
1. **Load and inspect the Custom Hybrid Model Architecture** (EVA-02 ViT branch + Deformable CNN branch + Cross-Attention Fusion).
2. **Initialize Stage Loaders** (using synthetic fallback mechanisms).
3. **Run Multi-task Inference** (Classification + Pixel-level Semantic Segmentation).
4. **Analyze the Adaptive Explainability (XAI) Engine** (Grad-CAM vs. Attention Rollout).
5. **Compute Quantitative XAI Metrics** (Faithfulness, Insertion AUC, Deletion AUC).

In [ ]:
import os
import sys
# Add project root to path
sys.path.append(os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt

from models.hybrid_model import AdaptiveExplainableHybridModel
from datasets import get_datasets
from xai.adaptive import AdaptiveXAISelector
from xai.metrics import compute_xai_metrics

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

### 1. Load Configurations and Model

In [ ]:
with open("configs/default_config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Force synthetic mode for demonstration
config['dataset']['synthetic'] = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Instantiating Hybrid Model...")
model = AdaptiveExplainableHybridModel(config).to(device)
model.eval()
print("Model parameter count:", sum(p.numel() for p in model.parameters() if p.requires_grad))

### 2. Prepare DataLoaders

In [ ]:
print("Loading Stage 2 (TACO classification and segmentation) loaders...")
train_loader, val_loader, test_loader = get_datasets(config, stage="stage2")

# Fetch a single sample batch
images, masks, labels = next(iter(val_loader))
print("Batch images shape:", images.shape)
print("Batch masks shape:", masks.shape)
print("Batch labels shape:", labels.shape)

### 3. Run Inference (Multi-task Prediction)

In [ ]:
# Select a single image
idx = 0
image = images[idx]
mask = masks[idx]
label = labels[idx].item()

image_batch = image.unsqueeze(0).to(device)

# Forward pass
with torch.set_grad_enabled(True): # Enable grads for CAM
    outputs = model(image_batch)

cls_logits = outputs['cls_logits'].detach().cpu()
seg_logits = outputs['seg_logits'].detach().cpu()

probs = torch.softmax(cls_logits, dim=1)[0]
pred_class_idx = torch.argmax(probs).item()
pred_class_name = config['dataset']['class_names'][pred_class_idx]
true_class_name = config['dataset']['class_names'][label]

print(f"True Category:      {true_class_name}")
print(f"Predicted Category: {pred_class_name} (Confidence: {probs[pred_class_idx].item():.4f})")

### 4. Generate Explainability Maps & Compute Quantitative XAI Metrics

In [ ]:
xai_selector = AdaptiveXAISelector(model, config)

# 1. Compute explainability maps
gradcam_map = xai_selector.get_explanation_by_method(image, method="gradcam", class_idx=pred_class_idx, device=device)
rollout_map = xai_selector.get_explanation_by_method(image, method="rollout", device=device)
adaptive_map, method_used, gating = xai_selector.get_explanation_by_gating(image, pred_class_idx, device=device)

print(f"Active Gating Selection: {method_used.upper()} (ViT={gating['alpha_vit']:.2f}, CNN={gating['alpha_cnn']:.2f})")

# 2. Calculate Faithfulness, Insertion, and Deletion AUC scores
print("Computing XAI metrics for Grad-CAM...")
gc_metrics = compute_xai_metrics(model, image, gradcam_map, pred_class_idx, grid_size=8, device=device)

print("Computing XAI metrics for Attention Rollout...")
ro_metrics = compute_xai_metrics(model, image, rollout_map, pred_class_idx, grid_size=8, device=device)

print(f"\n--- Explanation Metrics Summary ---")
print(f"Grad-CAM  -> Faithfulness: {gc_metrics['faithfulness']:.4f} | Insertion AUC: {gc_metrics['insertion_auc']:.4f} | Deletion AUC: {gc_metrics['deletion_auc']:.4f}")
print(f"Rollout   -> Faithfulness: {ro_metrics['faithfulness']:.4f} | Insertion AUC: {ro_metrics['insertion_auc']:.4f} | Deletion AUC: {ro_metrics['deletion_auc']:.4f}")

### 5. Plot Results

In [ ]:
# Denormalize image for plotting
img_np = image.cpu().numpy().transpose(1, 2, 0)
img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
img_np = np.clip(img_np, 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Input Image
axes[0].imshow(img_np)
axes[0].set_title(f"Input (GT: {true_class_name})", fontsize=12, weight='bold')
axes[0].axis('off')

# Segmentation prediction
pred_mask = torch.argmax(seg_logits, dim=1)[0].numpy()
axes[1].imshow(img_np)
axes[1].imshow(pred_mask, alpha=0.4, cmap='tab10')
axes[1].set_title(f"Segmentation (Pred: {pred_class_name})", fontsize=12, weight='bold')
axes[1].axis('off')

# Grad-CAM overlay
axes[2].imshow(img_np)
axes[2].imshow(gradcam_map, alpha=0.5, cmap='jet')
axes[2].set_title(f"Grad-CAM (Faith: {gc_metrics['faithfulness']:.2f})", fontsize=12, weight='bold')
axes[2].axis('off')

# Attention Rollout overlay
axes[3].imshow(img_np)
axes[3].imshow(rollout_map, alpha=0.5, cmap='jet')
axes[3].set_title(f"Rollout (Faith: {ro_metrics['faithfulness']:.2f})", fontsize=12, weight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.show()